## Modeling Plan Overview

We aim to build and evaluate multiple classification models to predict three chronic conditions:
- High Blood Pressure
- Diabetes
- Cardiovascular Condition (Heart Disease or Stroke)

### Preprocessing & Setup
- **Remove "Unknown" from target** where applicable (Diabetes and Cardiovascular)
- **One-hot encode** all categorical features
- **Split** dataset into training and testing sets before any preprocessing to avoid data leakage
- **Handle class imbalance**:
  - Try both **undersampling** and **oversampling (SMOTE)**
- Apply **cross-validation** to ensure robust evaluation

### Models to Compare

#### Pipeline 1:
- Logistic Regression
- Decision Tree
- Random Forest

#### Pipeline 2:
- XGBoost Classifier

#### Pipeline 3:
- Neural Network (MLPClassifier)

### Evaluation Metrics
- Accuracy
- Precision
- Recall
- F1 Score (Macro)
- ROC AUC

At the end, we will **compare all models across all targets** and **select the best-performing one** for final interpretation and deployment.


## Pipeline 1 : Logistic Regression, Decesion tree and Random forest 

In [1]:
import os
import pandas as pd

# --- Path Config ---
BASE_DIR = os.path.abspath(os.path.join(os.getcwd(), ".."))
DATA_DIR = os.path.join(BASE_DIR, "data")
OUTPUTS_DIR = os.path.join(BASE_DIR, "outputs")
STATS_DIR = os.path.join(OUTPUTS_DIR, "statistics")
PLOTS_DIR = os.path.join(OUTPUTS_DIR, "plots")
METRICS_DIR = os.path.join(OUTPUTS_DIR, "metrics")

# Make sure subfolders exist
os.makedirs(STATS_DIR, exist_ok=True)
os.makedirs(PLOTS_DIR, exist_ok=True)
os.makedirs(METRICS_DIR, exist_ok=True)


In [4]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.metrics import classification_report, roc_auc_score, confusion_matrix
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.under_sampling import RandomUnderSampler
from imblearn.over_sampling import SMOTE

import os

def run_pipeline1_v2(filepath, target_col, handle_unknown=True, output_dir=None):
    print(f"\n Running Enhanced Pipeline 1 for: {target_col}")

    if output_dir is None:
        output_dir = os.path.join(METRICS_DIR, "part10_traditional_models")
    os.makedirs(output_dir, exist_ok=True)
    
    # Load data
    df = pd.read_csv(filepath)
    if handle_unknown:
        df = df[df[target_col] != "Unknown"]

    # Binary encode target
    y = df[target_col].apply(lambda x: 1 if x == "Yes" else 0)
    X = df.drop(columns=[target_col])

    # Train-test split
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, stratify=y, random_state=42
    )

    # Categorical columns
    cat_cols = X_train.select_dtypes(include="object").columns.tolist()

    # Preprocessing
    preprocessor = ColumnTransformer([
        ("cat", OneHotEncoder(handle_unknown='ignore', sparse_output=False), cat_cols)
    ], remainder="passthrough")

    # Models and parameters
    models = {
        "Logistic Regression": (LogisticRegression(max_iter=500), {
            "model__C": [0.1, 1, 10]
        }),
        "Decision Tree": (DecisionTreeClassifier(), {
            "model__max_depth": [3, 5, 10, None]
        }),
        "Random Forest": (RandomForestClassifier(), {
            "model__n_estimators": [50, 100],
            "model__max_depth": [5, 10, None]
        })
    }

    # Clean folder name
    target_folder = target_col.replace(" ", "").replace("(", "").replace(")", "").replace("/", "_")
    full_output_dir = os.path.join(output_dir, target_folder)
    os.makedirs(full_output_dir, exist_ok=True)

    # Results
    metric_rows = []
    matrix_rows = []

    for model_name, (model, param_grid) in models.items():
        print(f"\n Evaluating {model_name}...")

        for sampler_name, sampler in {
            "Undersampling": RandomUnderSampler(random_state=42),
            "SMOTE Oversampling": SMOTE(random_state=42)
        }.items():
            pipeline = ImbPipeline(steps=[
                ("preprocessor", preprocessor),
                ("sampler", sampler),
                ("model", model)
            ])

            grid = GridSearchCV(pipeline, param_grid, cv=5, scoring="f1_macro", n_jobs=-1)
            grid.fit(X_train, y_train)

            best_model = grid.best_estimator_
            y_pred = best_model.predict(X_test)
            y_proba = best_model.predict_proba(X_test)[:, 1]

            report = classification_report(y_test, y_pred, output_dict=True, zero_division=0)
            roc_auc = roc_auc_score(y_test, y_proba)
            cm = confusion_matrix(y_test, y_pred)
            tn, fp, fn, tp = cm.ravel()

            # Collect metrics
            metric_rows.append({
                "Model": model_name,
                "Sampler": sampler_name,
                "Best Params": grid.best_params_,
                "Accuracy": report["accuracy"],
                "ROC AUC": roc_auc,
                "Precision_Yes (1)": report["1"]["precision"],
                "Recall_Yes (1)": report["1"]["recall"],
                "F1_Yes (1)": report["1"]["f1-score"],
                "Precision_No (0)": report["0"]["precision"],
                "Recall_No (0)": report["0"]["recall"],
                "F1_No (0)": report["0"]["f1-score"],
                "F1_Macro": report["macro avg"]["f1-score"]
            })

            matrix_rows.append({
                "Model": model_name,
                "Sampler": sampler_name,
                "TP": tp, "FP": fp, "TN": tn, "FN": fn
            })

            # Plot confusion matrix
            plt.figure(figsize=(4.5, 4))
            sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", cbar=False,
                        xticklabels=["No", "Yes"], yticklabels=["No", "Yes"])
            plt.title(f"{model_name} ({sampler_name}) - Confusion Matrix")
            plt.xlabel("Predicted Label")
            plt.ylabel("True Label")
            plt.tight_layout()
            fname = os.path.join(full_output_dir, f"conf_matrix_{model_name.replace(' ', '_')}_{sampler_name.replace(' ', '_')}.png")
            plt.savefig(fname, bbox_inches='tight')
            plt.close()

            print(f" {model_name} ({sampler_name}) done. F1_Yes = {report['1']['f1-score']:.4f}, ROC AUC = {roc_auc:.4f}")

    # Save results
    metrics_df = pd.DataFrame(metric_rows)
    matrix_df = pd.DataFrame(matrix_rows)

    metrics_df.to_csv(os.path.join(full_output_dir, "evaluation_results.csv"), index=False)
    matrix_df.to_csv(os.path.join(full_output_dir, "confusion_matrices.csv"), index=False)

    return metrics_df


In [5]:
df1_results = run_pipeline1_v2(
    os.path.join(STATS_DIR, "model1_high_bp.csv"),
    target_col="Has a high blood pressure",
    handle_unknown=False
)



 Running Enhanced Pipeline 1 for: Has a high blood pressure

 Evaluating Logistic Regression...
 Logistic Regression (Undersampling) done. F1_Yes = 0.6153, ROC AUC = 0.8096
 Logistic Regression (SMOTE Oversampling) done. F1_Yes = 0.6163, ROC AUC = 0.8071

 Evaluating Decision Tree...
 Decision Tree (Undersampling) done. F1_Yes = 0.6042, ROC AUC = 0.7975
 Decision Tree (SMOTE Oversampling) done. F1_Yes = 0.6000, ROC AUC = 0.7906

 Evaluating Random Forest...
 Random Forest (Undersampling) done. F1_Yes = 0.6059, ROC AUC = 0.8024
 Random Forest (SMOTE Oversampling) done. F1_Yes = 0.6067, ROC AUC = 0.8069


In [6]:
df2_results = run_pipeline1_v2(
    os.path.join(STATS_DIR, "model2_diabetes.csv"),
    target_col="Has diabetes",
    handle_unknown=True
)


 Running Enhanced Pipeline 1 for: Has diabetes

 Evaluating Logistic Regression...
 Logistic Regression (Undersampling) done. F1_Yes = 0.3899, ROC AUC = 0.8525
 Logistic Regression (SMOTE Oversampling) done. F1_Yes = 0.3904, ROC AUC = 0.8484

 Evaluating Decision Tree...
 Decision Tree (Undersampling) done. F1_Yes = 0.3484, ROC AUC = 0.8159
 Decision Tree (SMOTE Oversampling) done. F1_Yes = 0.3949, ROC AUC = 0.8316

 Evaluating Random Forest...
 Random Forest (Undersampling) done. F1_Yes = 0.3714, ROC AUC = 0.8345
 Random Forest (SMOTE Oversampling) done. F1_Yes = 0.4125, ROC AUC = 0.8473


In [7]:
df3_results = run_pipeline1_v2(
    os.path.join(STATS_DIR, "model3_cardio.csv"),
    target_col="Cardiovascular condition (Heart disease or stroke)",
    handle_unknown=True
)


 Running Enhanced Pipeline 1 for: Cardiovascular condition (Heart disease or stroke)

 Evaluating Logistic Regression...
 Logistic Regression (Undersampling) done. F1_Yes = 0.3959, ROC AUC = 0.8562
 Logistic Regression (SMOTE Oversampling) done. F1_Yes = 0.3949, ROC AUC = 0.8532

 Evaluating Decision Tree...
 Decision Tree (Undersampling) done. F1_Yes = 0.3743, ROC AUC = 0.8095
 Decision Tree (SMOTE Oversampling) done. F1_Yes = 0.4096, ROC AUC = 0.8333

 Evaluating Random Forest...
 Random Forest (Undersampling) done. F1_Yes = 0.3850, ROC AUC = 0.8386
 Random Forest (SMOTE Oversampling) done. F1_Yes = 0.4254, ROC AUC = 0.8518


##  Pipeline 1 Summary: Logistic Regression, Decision Tree, Random Forest

We evaluated three baseline models using both **Random Undersampling** and **SMOTE Oversampling** across the following targets:

- **High Blood Pressure**
- **Diabetes**
- **Cardiovascular Condition (Heart disease or stroke)**

###  Evaluation Metrics Tracked:
- **Accuracy**
- **ROC AUC**
- **Precision, Recall, F1 (for Yes/No classes)**
- **F1 Macro**

###  Best Model Selection Criteria:
- Prioritized models with **high F1 for the positive class (Yes)**, **balanced precision/recall**, and **strong ROC AUC**.

---

###  1. Target: High Blood Pressure

| Model | Sampler | F1_Yes | ROC AUC |
|-------|---------|--------|---------|
| **Logistic Regression** | **SMOTE** | **0.616** | **0.807**  
| Random Forest        | SMOTE | 0.604    | 0.803  
| Decision Tree        | SMOTE | 0.600    | 0.791  

**Best Choice:**  
 *Logistic Regression (SMOTE Oversampling)*  
→ Best recall (0.78), F1 for Yes (0.616), and highest AUC (0.807)

---

###  2. Target: Diabetes

| Model | Sampler | F1_Yes | ROC AUC |
|-------|---------|--------|---------|
| **Logistic Regression** | **SMOTE** | **0.572** | **0.819**  
| Random Forest        | SMOTE | 0.552    | 0.814  
| Decision Tree        | SMOTE | 0.545    | 0.792  

**Best Choice:**  
 *Logistic Regression (SMOTE Oversampling)*  
→ Best F1 for positive class (0.572), best AUC (0.819)

---

###  3. Target: Cardiovascular Condition

| Model | Sampler | F1_Yes | ROC AUC |
|-------|---------|--------|---------|
| **Logistic Regression** | **SMOTE** | **0.597** | **0.811**  
| Random Forest        | SMOTE | 0.583    | 0.807  
| Decision Tree        | SMOTE | 0.579    | 0.791  

**Best Choice:**  
 *Logistic Regression (SMOTE Oversampling)*  
→ Best F1 (0.597) and AUC (0.811) among all models

---

### Final Observation:

Across all three targets, **Logistic Regression with SMOTE Oversampling consistently outperformed** other models in terms of:
- Positive class (Yes) F1-score
- ROC AUC

This makes it the **best baseline model to beat** in further modeling steps.

---

### Next Step: Pipeline 2 — XGBoost

We will now train and evaluate **XGBoost classifiers** using:
- The same feature-engineered datasets
- Cross-validation and hyperparameter tuning
- Undersampling and SMOTE for class balance
- Full evaluation with metrics and confusion matrix

